# Importing and Preparing

In [2]:
from __future__ import annotations

import argparse
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

## Defining Basic Constants

In [3]:
RAW_FILES = {  # logical name -> raw file name (note the raw typo "oppenent")
    "players_stats": "epl_players_stats_2021_2026.csv",
    "players_shooting": "epl_players_shooting_2021_2026.csv",
    "keepers_stats": "epl_keepers_stats_2021_2026.csv",
    "squad_stats": "epl_squad_stats_2021_2026.csv",
    "squad_shooting": "epl_squad_shooting_2021_2026.csv",
    "squad_goalkeeping": "epl_squad_goalkeeping_2021_2026.csv",
    "squad_misc": "epl_squad_misc_2021_2026.csv",
    "squad_playingtime": "epl_squad_playingtime_2021_2026.csv",
    "opponent_stats": "epl_opponent_stats_2021_2026.csv",
    "opponent_shooting": "epl_oppenent_shooting_2021_2026.csv",
    "opponent_goalkeeping": "epl_oppenent_goalkeeping_2021_2026.csv",
    "opponent_misc": "epl_oppenent_misc_2021_2026.csv",
    "opponent_playingtime": "epl_oppenent_playingtime_2021_2026.csv",
    "rank_overall": "epl_rank_overall_2021_2026.csv",
    "rank_home_away": "epl_rank_home_away_2021_2026.csv",
}
MATCHES_OUT = "epl_matches_2021_2026.csv"
MATCHES_ODDS_OUT = "epl_matches_odds_extra_2021_2026.csv"

PEOPLE = ["players_stats", "players_shooting", "keepers_stats"]
TEAM_TABLES = [k for k in RAW_FILES if k not in PEOPLE]
OPPONENT_TABLES = [k for k in RAW_FILES if k.startswith("opponent_")]
EXPECTED_TEAMS_PER_SEASON = 20
ODDS_MIN_FILL = 0.95   # an odds column is kept in the main matches file only if >= 95% filled in EVERY season

# Matches.csv name -> name used in all the FBref-style files
MATCHES_TEAM_MAP = {
    "Man City": "Manchester City", "Man United": "Manchester Utd", "Nott'm Forest": "Nottingham",
    "Leeds": "Leeds United", "Leicester": "Leicester City", "Norwich": "Norwich City",
    "Ipswich": "Ipswich Town", "Luton": "Luton Town",
}
# readable names for the match-result / match-stat columns (odds columns get generic snake_case)
MATCHES_RENAME = {
    "HomeTeam": "home_team", "AwayTeam": "away_team",
    "FTHG": "ft_home_goals", "FTAG": "ft_away_goals", "FTR": "ft_result",
    "HTHG": "ht_home_goals", "HTAG": "ht_away_goals", "HTR": "ht_result",
    "HS": "home_shots", "AS": "away_shots", "HST": "home_shots_on_target", "AST": "away_shots_on_target",
    "HF": "home_fouls", "AF": "away_fouls", "HC": "home_corners", "AC": "away_corners",
    "HY": "home_yellows", "AY": "away_yellows", "HR": "home_reds", "AR": "away_reds",
}
MATCH_CORE = ["Season", "Date", "Time", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "HTHG", "HTAG",
              "HTR", "Referee", "HS", "AS", "HST", "AST", "HF", "AF", "HC", "AC", "HY", "AY", "HR", "AR"]

## Context Helper Class

In [ ]:
class Ctx:
    """Holds the data frames and the audit trail while the pipeline runs."""

    def __init__(self) -> None:
        self.frames: dict[str, pd.DataFrame] = {}
        self.orig_cols: dict[str, list[str]] = {}
        self.dropped: dict[str, list[tuple[str, str]]] = {}
        self.log: list[dict] = []
        self.canonical_teams: set[str] = set()

    def note(self, step: int, dataset: str, action: str, n: int | None = None, detail: str = "") -> None:
        self.log.append({"step": step, "dataset": dataset, "action": action, "rows_affected": n, "detail": detail})

    def drop(self, name: str, cols: list[str], reason: str) -> None:
        df = self.frames[name]
        present = [c for c in cols if c in df.columns]
        self.frames[name] = df.drop(columns=present)
        for c in present:
            self.dropped.setdefault(name, []).append((c, reason))

    def summary(self) -> pd.DataFrame:
        """Returns a high-level overview table of all datasets currently loaded."""
        data = []
        for name, df in self.frames.items():
            data.append({
                "dataset": name,
                "rows": df.shape[0],
                "columns": df.shape[1],
                "memory_kb": round(df.memory_usage(deep=True).sum() / 1024, 2)
            })
        if not data:
            return pd.DataFrame(columns=["dataset", "rows", "columns", "memory_kb"])
        return pd.DataFrame(data)

    def show_log(self, step: int | None = None, dataset: str | None = None) -> pd.DataFrame:
        """Returns the pipeline audit trail, optionally filtered by step or dataset."""
        if not self.log:
            return pd.DataFrame(columns=["step", "dataset", "action", "rows_affected", "detail"])
        
        filtered = self.log
        if step is not None:
            filtered = [entry for entry in filtered if entry["step"] == step]
        if dataset is not None:
            filtered = [entry for entry in filtered if entry["dataset"] == dataset]
            
        return pd.DataFrame(filtered)

    def show_dropped(self, dataset: str | None = None) -> pd.DataFrame:
        """Flattens dropped columns into an audit table, optionally filtered by dataset."""
        rows = []
        for ds, cols in self.dropped.items():
            if dataset is not None and ds != dataset:
                continue
            for col, reason in cols:
                rows.append({"dataset": ds, "column": col, "reason": reason})
        if not rows:
            return pd.DataFrame(columns=["dataset", "column", "reason"])
        return pd.DataFrame(rows)

    def show_teams(self) -> pd.DataFrame:
        """Returns the set of canonical teams as a sorted DataFrame."""
        if not self.canonical_teams:
            return pd.DataFrame(columns=["canonical_team"])
        return pd.DataFrame({"canonical_team": sorted(list(self.canonical_teams))})